In [1]:
import os
import re
import json
import faiss
import pickle
import sqlite3
import numpy as np
from typing import List, Dict, Any, Literal, Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END, START
from langgraph.graph.state import CompiledStateGraph
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
import importlib
import grandalf
importlib.reload(grandalf)


from sentence_transformers import SentenceTransformer, CrossEncoder



c:\Users\Lenovo\Desktop\rag\rrr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DB_PATH          = r"C:/Users/Lenovo/Desktop/rag/sales.db"
FAISS_STORE_DIR  = "adaptive_langgraph_faiss"
EMBEDDING_MODEL  = "all-MiniLM-L6-v2"
RERANKER_MODEL   = "BAAI/bge-reranker-base"
OLLAMA_MODEL     = "llama3.2"   # غيّره لو عندك موديل تاني (llama3.1, mistral, ...)
MAX_LOOPS        = 2            # أقصى عدد مرات الـ Reflection
TOP_K_INITIAL    = 20
TOP_K_FINAL      = 5

print(f"✅ Done| Ollama Model: {OLLAMA_MODEL}")

✅ Done| Ollama Model: llama3.2


In [3]:
class GraphState(TypedDict):
    """
    الـ State اللي بيتنقل بين كل nodes في الـ Graph
    """
    
    query: str                 
    route: str                         
    improved_query: str                 
    alternative_queries: List[str]     
    retrieved_docs: List[Dict]          
    context: str                        
    answer: str                     
    reflection_score: int               
    reflection_feedback: str            
    loops: int                          
print("✅ GraphState!")

✅ GraphState!


In [4]:

def load_from_db(db_path: str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM products")
    rows = cursor.fetchall()
    conn.close()
    
    docs = []
    for row in rows:
        content = " | ".join([f"{k}: {row[k]}" for k in row.keys()])
        docs.append(Document(page_content=content, metadata=dict(row)))
    
    print(f"✅ تم تحميل {len(docs)} منتج")
    return docs


# ─── الـ Vector Store ─────────────────────────────────────────
class FaissStore:
    def __init__(self, embed_model: str, rerank_model: str, store_dir: str):
        self.store_dir = store_dir
        os.makedirs(store_dir, exist_ok=True)
        self.index = None
        self.metadata = []
        
        print("[INFO] تحميل موديلات الـ Embedding والـ Reranker...")
        self.embedder = SentenceTransformer(embed_model)
        self.reranker = CrossEncoder(rerank_model)
        print("✅ الـ Vector Store جاهز!")
    
    def build(self, docs: List[Document]):
        import hashlib
        texts = [doc.page_content for doc in docs]
        embeddings = self.embedder.encode(texts, show_progress_bar=True)
        
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(np.array(embeddings).astype('float32'))
        self.metadata = [{"text": doc.page_content} for doc in docs]
        
        
        all_hashes = {
            hashlib.md5(doc.page_content.encode()).hexdigest()
            for doc in docs
        }
        with open(f"{self.store_dir}/hashes.pkl", "wb") as f:
            pickle.dump(all_hashes, f)
        
        self._save()
        print(f"✅ Index جاهز! ({len(docs)} منتج)")
    
    def _save(self):
        faiss.write_index(self.index, f"{self.store_dir}/faiss.index")
        with open(f"{self.store_dir}/metadata.pkl", "wb") as f:
            pickle.dump(self.metadata, f)
    
    def load(self):
        self.index = faiss.read_index(f"{self.store_dir}/faiss.index")
        with open(f"{self.store_dir}/metadata.pkl", "rb") as f:
            self.metadata = pickle.load(f)
        print(f"✅ Index محمّل ({len(self.metadata)} chunks)")
    
    def update(self, docs: List[Document]):
        import hashlib
        
        # جيب الـ hashes القديمة
        hashes_path = f"{self.store_dir}/hashes.pkl"
        if os.path.exists(hashes_path):
            with open(hashes_path, "rb") as f:
                old_hashes = pickle.load(f)
        else:
            old_hashes = set()
        
        
        new_docs = [
            doc for doc in docs
            if hashlib.md5(doc.page_content.encode()).hexdigest() not in old_hashes
        ]
        
        if not new_docs:
            print("✅ مفيش جديد، الـ Index محدّث!")
            return
        
        print(f"[INFO] {len(new_docs)} منتج جديد، جاري الإضافة...")
        
        
        new_texts = [doc.page_content for doc in new_docs]
        new_embeddings = self.embedder.encode(new_texts, show_progress_bar=True)
        
       
        self.index.add(np.array(new_embeddings).astype('float32'))
        
        
        self.metadata.extend([{"text": doc.page_content} for doc in new_docs])
        
        
        self._save()
        all_hashes = old_hashes | {
            hashlib.md5(doc.page_content.encode()).hexdigest()
            for doc in docs
        }
        with open(hashes_path, "wb") as f:
            pickle.dump(all_hashes, f)
        
        print(f"✅ تمت الإضافة! ({len(new_docs)} منتج جديد)")
    
    def search(self, query: str, top_k_init: int = 20, top_k_final: int = 5) -> List[Dict]:
        emb = self.embedder.encode([query]).astype('float32')
        D, I = self.index.search(emb, top_k_init)
        
        candidates = [
            self.metadata[idx]["text"]
            for idx in I[0] if 0 <= idx < len(self.metadata)
        ]
        if not candidates:
            return []
        
        scores = self.reranker.predict([[query, t] for t in candidates])
        ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        
        return [{"text": t, "score": float(s)} for t, s in ranked[:top_k_final]]
    
    def multi_search(self, queries: List[str], top_k_final: int = 5) -> List[Dict]:
        seen, merged = set(), []
        for q in queries:
            for r in self.search(q, top_k_init=10, top_k_final=10):
                key = r["text"][:80]
                if key not in seen:
                    seen.add(key)
                    merged.append(r)
        
        if not merged:
            return []
        scores = self.reranker.predict([[queries[0], r["text"]] for r in merged])
        ranked = sorted(zip(merged, scores), key=lambda x: x[1], reverse=True)
        return [{**r, "score": float(s)} for r, s in ranked[:top_k_final]]


db_docs = load_from_db(DB_PATH)





vector_store = FaissStore(EMBEDDING_MODEL, RERANKER_MODEL, FAISS_STORE_DIR)


index_exists = os.path.exists(f"{FAISS_STORE_DIR}/faiss.index")

if index_exists:
    print("✅ Index موجود، جاري التحميل...")
    vector_store.load()
    vector_store.update(db_docs)  # ← ضيف السطر ده
else:
    print("🔨 Index مش موجود، جاري البناء...")
    vector_store.build(db_docs)


✅ تم تحميل 650 منتج
[INFO] تحميل موديلات الـ Embedding والـ Reranker...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1002.97it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1137.50it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]             
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ الـ Vector Store جاهز!
✅ Index موجود، جاري التحميل...
✅ Index محمّل (1150 chunks)
✅ مفيش جديد، الـ Index محدّث!


In [5]:
llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)


llm_json = ChatOllama(model=OLLAMA_MODEL, temperature=0, format="json")


print("🧪 اختبار الاتصال بـ Ollama...")
test = llm.invoke("قل كلمة واحدة فقط: مرحبا")
print(f"✅ Ollama يعمل! الرد: {test.content}")

🧪 اختبار الاتصال بـ Ollama...
✅ Ollama يعمل! الرد: مرحبا


In [6]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def node_query_router(state: GraphState) -> GraphState:
    print("\n📍 [NODE] query_router")
    
    prompt = ChatPromptTemplate.from_template("""
You are a routing system for a products and sales database.
Analyze the question and respond with JSON only.

Question: {query}

Respond in this format:
{{"route": "direct" or "simple" or "complex"}}

Rules:
- direct:  greeting or general question that doesn't need database
- simple:  question about one product or specific information
- complex: comparison or analysis or needs multiple products
""")
    
    
    
    chain = prompt | llm_json | JsonOutputParser()
    
    try:
        result = chain.invoke({"query": state["query"]})
        route = result.get("route", "simple")
        if route not in ["direct", "simple", "complex"]:
            route = "simple"
    except Exception as e:
        print(f"   ⚠️ خطأ في الـ Router: {e} → fallback to simple")
        route = "simple"
    
    print(f"   ➡️ Route: {route}")
    return {**state, "route": route, "loops": 0}



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def node_query_rewriter(state: GraphState) -> GraphState:
    print("\n📍 [NODE] query_rewriter")
    
    feedback = state.get("reflection_feedback", "")
    feedback_part = f"\nFeedback from previous evaluation: {feedback}" if feedback else ""
    
    prompt = ChatPromptTemplate.from_template("""
You are a search query optimizer for a products database.
The database contains: product name, category, price, quantity, rating.

Important rules:
- Never write SQL
- Only write natural search keywords
- Keep it simple and specific

Original question: {query}{feedback_part}

Reply in JSON only:
{{"improved_query": "optimized search keywords", "alternative_queries": ["alternative 1", "alternative 2"]}}

Example:
Question: "what is the price of Milk Semi-Skimmed 1L?"
Answer: {{"improved_query": "Milk Semi-Skimmed 1L price", "alternative_queries": ["Milk Semi-Skimmed", "1L milk"]}}
""")
    
    chain = prompt | llm_json | JsonOutputParser()
    
    try:
        result = chain.invoke({"query": state["query"], "feedback_part": feedback_part})
        improved = result.get("improved_query", state["query"])
        alternatives = result.get("alternative_queries", [state["query"]])
    except Exception as e:
        print(f"   ⚠️ خطأ في الـ Rewriter: {e} → fallback")
        improved = state["query"]
        alternatives = [state["query"]]
    
    print(f"   ✏️ السؤال المحسّن: {improved}")
    return {**state, "improved_query": improved, "alternative_queries": alternatives}



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def node_retriever_simple(state: GraphState) -> GraphState:
    print("\n📍 [NODE] retriever_simple")
    
    query = state.get("improved_query", state["query"])
    results = vector_store.search(query, TOP_K_INITIAL, TOP_K_FINAL)
    
    context = "\n\n".join([f"[{i+1}] {r['text']}" for i, r in enumerate(results)])
    
    print(f"   🔍 find{len(results)} Results")
    return {**state, "retrieved_docs": results, "context": context}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def node_retriever_multi(state: GraphState) -> GraphState:
    print("\n📍 [NODE] retriever_multi (Multi-Step)")
    
    improved = state.get("improved_query", state["query"])
    alternatives = state.get("alternative_queries", [])
    all_queries = [improved] + alternatives[:2]  
    
    print(f"   🔍 البحث بـ {len(all_queries)} استعلام")
    results = vector_store.multi_search(all_queries, TOP_K_FINAL)
    
    context = "\n\n".join([f"[{i+1}] {r['text']}" for i, r in enumerate(results)])
    
    print(f"   🔍 وجد {len(results)} نتيجة مدمجة")
    return {**state, "retrieved_docs": results, "context": context}



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def node_generator(state: GraphState) -> GraphState:
    print("\n📍 [NODE] generator")
    
    if not state.get("retrieved_docs"):
        return {**state, "answer": "Sorry, I couldn't find enough information in the database."}
    
    formatted_docs = []
    for i, r in enumerate(state["retrieved_docs"]):
        text = r["text"]
        formatted_docs.append(f"""
Product {i+1}:
{text.replace(" | ", chr(10))}
""")
    
    context = "\n---\n".join(formatted_docs)
    
    prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant that answers questions about products.
The following data is taken directly from the database.
Each line represents a column and its value.
Use the data exactly as it is to answer the question.

Question: {query}

Data:
{context}

Answer (use the exact numbers and values from the data above):
""")
    
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"query": state["query"], "context": context})
    
    print(f"   💬 الإجابة جاهزة ({len(answer)} حرف)")
    return {**state, "answer": answer, "context": context}




# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def node_direct_answer(state: GraphState) -> GraphState:
    print("\n📍 [NODE] direct_answer")
    
    prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant specialized in products and sales data.
Answer the following question:
{query}
""")
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"query": state["query"]})
    
    print(f"   💬 إجابة مباشرة ({len(answer)} حرف)")
    return {**state, "answer": answer, "retrieved_docs": [], "context": ""}



# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def node_reflector(state: GraphState) -> GraphState:
    print("\n📍 [NODE] reflector")
    
    prompt = ChatPromptTemplate.from_template("""
Evaluate the quality of the following answer and respond in JSON:

Question: {query}
Answer: {answer}
Context used: {context}

Reply in JSON only:
{{"score": number from 1 to 10, "sufficient": true or false, "feedback": "your notes", "new_search_query": "new search query if sufficient=false"}}

Consider the answer sufficient if score >= 6 or if the answer reasonably answered the question.
""")
    
    chain = prompt | llm_json | JsonOutputParser()
    
    try:
        result = chain.invoke({
            "query": state["query"],
            "answer": state["answer"],
            "context": state.get("context", "")[:1000]
        })
        score = int(result.get("score", 7))
        sufficient = result.get("sufficient", True)
        feedback = result.get("feedback", "")
        new_query = result.get("new_search_query", "")
    except Exception as e:
        print(f"   ⚠️ خطأ في الـ Reflector: {e} → اعتبرها كافية")
        score, sufficient, feedback, new_query = 7, True, "", ""
    
    print(f"   🪞 Score: {score}/10 | Sufficient: {sufficient}")
    if feedback:
        print(f"   📝 Feedback: {feedback}")
    
    # لو في استعلام جديد، حدّث السؤال للـ Loop القادم
    updated_query = new_query if new_query and not sufficient else state["query"]
    
    return {
        **state,
        "reflection_score": score,
        "reflection_feedback": feedback,
        "query": updated_query,   # هيتحسن في الـ rewriter القادم
        "loops": state.get("loops", 0) + 1
    }


print("✅ تم تعريف كل الـ Nodes!")

✅ تم تعريف كل الـ Nodes!


In [7]:

def edge_after_router(state: GraphState) -> Literal["direct", "rewrite_simple", "rewrite_complex"]:
    route = state.get("route", "simple")
    if route == "direct":
        return "direct"
    elif route == "complex":
        return "rewrite_complex"
    else:
        return "rewrite_simple"



def edge_after_reflection(state: GraphState) -> Literal["end", "retry"]:
    sufficient = state.get("reflection_score", 7) >= 6
    loops_done = state.get("loops", 0)
    
    if loops_done >= MAX_LOOPS:
        print(f"   ⚠️ وصلنا الحد الأقصى ({MAX_LOOPS} loops) → END")
        return "end"
    
    if sufficient:
        print(f"   ✅ الإجابة كويسة → END")
        return "end"
    else:
        print(f"   🔄 الإجابة تحتاج تحسين (loop {loops_done}/{MAX_LOOPS}) → RETRY")
        return "retry"


print("✅ تم تعريف الـ Conditional Edges!")

✅ تم تعريف الـ Conditional Edges!


In [8]:


workflow = StateGraph(GraphState)


workflow.add_node("query_router",      node_query_router)
workflow.add_node("query_rewriter",    node_query_rewriter)
workflow.add_node("retriever_simple",  node_retriever_simple)
workflow.add_node("retriever_multi",   node_retriever_multi)
workflow.add_node("generator",         node_generator)
workflow.add_node("direct_answer",     node_direct_answer)
workflow.add_node("reflector",         node_reflector)


workflow.add_edge(START, "query_router")


workflow.add_conditional_edges(
    "query_router",
    edge_after_router,
    {
        "direct":          "direct_answer",
        "rewrite_simple":  "query_rewriter",
        "rewrite_complex": "query_rewriter",
    }
)


workflow.add_conditional_edges(
    "query_rewriter",
    lambda s: "multi" if s.get("route") == "complex" else "simple",
    {
        "simple": "retriever_simple",
        "multi":  "retriever_multi",
    }
)


workflow.add_edge("retriever_simple", "generator")
workflow.add_edge("retriever_multi",  "generator")


workflow.add_edge("generator", "reflector")


workflow.add_edge("direct_answer", END)


workflow.add_conditional_edges(
    "reflector",
    edge_after_reflection,
    {
        "end":   END,
        "retry": "query_rewriter",   
    }
)


app = workflow.compile()


print(app.get_graph().draw_ascii())

                                              +-----------+                                               
                                              | __start__ |                                               
                                              +-----------+                                               
                                                     *                                                    
                                                     *                                                    
                                                     *                                                    
                                             +--------------+                                             
                                             | query_router |...                                          
                                             +--------------+   .......                                   
                                     

In [9]:
def ask(question: str, verbose: bool = True) -> str:
    """دالة مبسّطة لاستخدام الـ Graph"""
    
    print(f"\n{'='*60}")
    print(f"❓ السؤال: {question}")
    print(f"{'='*60}")
    
    initial_state: GraphState = {
        "query": question,
        "route": "",
        "improved_query": "",
        "alternative_queries": [],
        "retrieved_docs": [],
        "context": "",
        "answer": "",
        "reflection_score": 0,
        "reflection_feedback": "",
        "loops": 0,
    }
    
    final_state = app.invoke(initial_state)
    
    print(f"\n{'─'*60}")
    print("💬 الإجابة النهائية:")
    print(final_state["answer"])
    print(f"\n📊 Route: {final_state.get('route','?')} | Loops: {final_state.get('loops',0)} | Score: {final_state.get('reflection_score','?')}")
    
    return final_state["answer"]

In [ ]:
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
import threading
import time


class DBWatcher(FileSystemEventHandler):
    def on_modified(self, event):
        if "sales.db" in event.src_path:
            print("🔄 DB اتغيّرت، جاري التحديث...")
            
            # عرّف الـ function هنا جوا
            import sqlite3
            conn = sqlite3.connect(DB_PATH)
            conn.row_factory = sqlite3.Row
            cursor = conn.cursor()
            cursor.execute("SELECT * FROM products")
            rows = cursor.fetchall()
            conn.close()
            
            docs = [
                Document(
                    page_content=" | ".join([f"{k}: {row[k]}" for k in row.keys()]),
                    metadata=dict(row)
                )
                for row in rows
            ]
            
            vector_store.update(docs)
            print("✅ التحديث خلص!")

observer = Observer()
observer.schedule(
    DBWatcher(),
    path="C:/Users/Lenovo/Desktop/rag/",
    recursive=False
)
observer.start()

# خلّيه شغال في الخلفية
try:
    while True:
        time.sleep(5)  # بيفحص كل 5 ثواني
except KeyboardInterrupt:
    observer.stop()
    
observer.join()


In [10]:

ask("id  ايه هو Baby Car Seat Group 0؟")


❓ السؤال: id  ايه هو Baby Car Seat Group 0؟

📍 [NODE] query_router
   ➡️ Route: simple

📍 [NODE] query_rewriter
   ✏️ السؤال المحسّن: Baby Car Seat Group 0

📍 [NODE] retriever_simple
   🔍 find5 Results

📍 [NODE] generator
   💬 الإجابة جاهزة (290 حرف)

📍 [NODE] reflector
   🪞 Score: 8/10 | Sufficient: True
   📝 Feedback: The answer provided the necessary information about Product ID 650, including its name, supplier ID, category ID, quantity per unit, price, and stock levels. However, it would be more helpful if the answer also included a brief description of what 'Baby Car Seat Group 0' is, as this term might be unfamiliar to some users.
   ✅ الإجابة كويسة → END

────────────────────────────────────────────────────────────
💬 الإجابة النهائية:
The product with id 650 is a Baby Car Seat Group 0, which has the following specifications:

- Product Name: Baby Car Seat Group 0
- Supplier ID: 35
- Category ID: 16
- Quantity per unit: 1 unit
- Price: $120.00
- Units in stock: 10
- Units on or

'The product with id 650 is a Baby Car Seat Group 0, which has the following specifications:\n\n- Product Name: Baby Car Seat Group 0\n- Supplier ID: 35\n- Category ID: 16\n- Quantity per unit: 1 unit\n- Price: $120.00\n- Units in stock: 10\n- Units on order: 0\n- Reorder level: 2\n- Discontinued: No'

In [ ]:
# ─── مثال 3: سؤال مباشر (بدون بحث) ──────────────────────────
ask("مرحبا، كيف يمكنني الاستفادة منك؟")

In [ ]:
# ─── مثال 2: سؤال مركب ───────────────────────────────────────
ask("قارن بين أغلى 3 منتجات في قاعدة البيانات وأيها الأفضل تقييماً؟")